"""
Phase 4 -- Convert graph artifacts (features, edges, labels) into a
PyTorch Geometric `Data` object, with train/val/test as NODE MASKS
rather than separate row splits (transductive node classification
setup).
"""

In [1]:
import numpy as np
import pandas as pd
import torch
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split

In [2]:
X_PATH = "Data/X_features.csv"
Y_PATH = "Data/y_labels.csv"
EDGE_PATH = "Output/edge_index.npy"
OUT_PATH = "Output/graph_data.pt"

---------------------------------------------------------------
Step 1: Load Phase 2/3 artifacts and convert to tensors.
---------------------------------------------------------------

In [3]:
X = pd.read_csv(X_PATH).values.astype(np.float32)
y = pd.read_csv(Y_PATH).values.ravel().astype(np.int64)
edge_index_np = np.load(EDGE_PATH)

x = torch.tensor(X, dtype=torch.float)                  # [num_nodes, num_features]
edge_index = torch.tensor(edge_index_np, dtype=torch.long)  # [2, num_edges]
y_tensor = torch.tensor(y, dtype=torch.long)             # [num_nodes]

print(f"x: {x.shape}, edge_index: {edge_index.shape}, y: {y_tensor.shape}")

x: torch.Size([704, 42]), edge_index: torch.Size([2, 10048]), y: torch.Size([704])


---------------------------------------------------------------
Step 2: Build the PyG Data object.
---------------------------------------------------------------

In [4]:
data = Data(x=x, edge_index=edge_index, y=y_tensor)
print(f"\nPyG Data object: {data}")
print(f"Is directed: {data.is_directed()}")  # should be False -- we
                                              # included both edge
                                              # directions in Phase 3
print(f"Contains isolated nodes: {data.has_isolated_nodes()}")
print(f"Contains self-loops: {data.has_self_loops()}")


PyG Data object: Data(x=[704, 42], edge_index=[2, 10048], y=[704])
Is directed: False
Contains isolated nodes: False
Contains self-loops: False


---------------------------------------------------------------
Step 3: Create stratified train/val/test NODE MASKS.
Why stratified: our classes are imbalanced (73% NO / 27% YES from
Phase 1). A plain random split could, by bad luck, put very few
ASD-positive nodes in the val or test set, making those metrics
unreliable. Stratifying preserves the ~73/27 ratio in every split.
Why masks, not separate arrays: the graph (x, edge_index) stays
ONE single object. A mask is just a boolean vector of length
num_nodes marking which nodes' labels are "visible" for a given
phase (training loss vs validation/test evaluation). This is the
standard PyG pattern for transductive node classification.


In [5]:
num_nodes = data.num_nodes
indices = np.arange(num_nodes)

First split off the test set (15%), stratified by label

In [6]:
train_val_idx, test_idx = train_test_split(
    indices, test_size=0.15, stratify=y, random_state=42
)

Then split remaining 85% into train (70% of total) / val (15% of total)
0.15 / 0.85 ≈ 0.1765 so val ends up ~15% of the ORIGINAL total

In [7]:
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=0.1765, stratify=y[train_val_idx], random_state=42
)

def indices_to_mask(idx_array, num_nodes):
    mask = torch.zeros(num_nodes, dtype=torch.bool)
    mask[idx_array] = True
    return mask

data.train_mask = indices_to_mask(train_idx, num_nodes)
data.val_mask = indices_to_mask(val_idx, num_nodes)
data.test_mask = indices_to_mask(test_idx, num_nodes)

--------------------------------------------------------------
Step 4: Sanity checks -- ALWAYS verify mask integrity before
trusting any downstream training run. A silent overlap bug here
would quietly leak test labels into training.
---------------------------------------------------------------

In [8]:
print("\n" + "=" * 60)
print("MASK SANITY CHECKS")
print("=" * 60)
n_train, n_val, n_test = data.train_mask.sum().item(), data.val_mask.sum().item(), data.test_mask.sum().item()
print(f"Train nodes: {n_train} ({100*n_train/num_nodes:.1f}%)")
print(f"Val nodes:   {n_val} ({100*n_val/num_nodes:.1f}%)")
print(f"Test nodes:  {n_test} ({100*n_test/num_nodes:.1f}%)")
print(f"Total accounted for: {n_train + n_val + n_test} / {num_nodes}")

overlap_train_val = (data.train_mask & data.val_mask).sum().item()
overlap_train_test = (data.train_mask & data.test_mask).sum().item()
overlap_val_test = (data.val_mask & data.test_mask).sum().item()
print(f"\nOverlap train/val: {overlap_train_val} (must be 0)")
print(f"Overlap train/test: {overlap_train_test} (must be 0)")
print(f"Overlap val/test: {overlap_val_test} (must be 0)")
assert overlap_train_val == overlap_train_test == overlap_val_test == 0, "MASK OVERLAP BUG!"
assert (n_train + n_val + n_test) == num_nodes, "Not all nodes accounted for!"

print("\nClass balance per split (should all be ~73/27):")
for name, mask in [("train", data.train_mask), ("val", data.val_mask), ("test", data.test_mask)]:
    split_y = data.y[mask]
    pos_ratio = split_y.float().mean().item()
    print(f"  {name:5s}: n={mask.sum().item():3d}, ASD-positive ratio={pos_ratio:.3f}")


MASK SANITY CHECKS
Train nodes: 492 (69.9%)
Val nodes:   106 (15.1%)
Test nodes:  106 (15.1%)
Total accounted for: 704 / 704

Overlap train/val: 0 (must be 0)
Overlap train/test: 0 (must be 0)
Overlap val/test: 0 (must be 0)

Class balance per split (should all be ~73/27):
  train: n=492, ASD-positive ratio=0.268
  val  : n=106, ASD-positive ratio=0.274
  test : n=106, ASD-positive ratio=0.264


---------------------------------------------------------------
Step 5: Save for Phase 5 (model building/training)
---------------------------------------------------------------

In [9]:
torch.save(data, OUT_PATH)
print(f"\nSaved PyG Data object with masks to {OUT_PATH}")
print(f"\nFinal object: {data}")


Saved PyG Data object with masks to Output/graph_data.pt

Final object: Data(x=[704, 42], edge_index=[2, 10048], y=[704], train_mask=[704], val_mask=[704], test_mask=[704])
